# Computing bijective correspondences by minimizing the Ginzburg-Landau energy

This notebook showcases the basic usage of the method: given two genus-zero surfaces
$A$ and $B$ and a (possibly low-quality) input map between them, we compute a
continuous, bijective, orientation-preserving correspondence.

The algorithm has four steps, mirrored below:

1. **Discrete Complex Line Bundle Structure**:  build a discrete connection on each surface with total
   curvature $2\pi$, so that each slice of the product space carries a single zero.
2. **Initialization**: turn the input map into an initial section $z$ on $A \times B$.
3. **GL minimization**: minimize the discrete Ginzburg-Landau energy of $z$.
4. **Evaluation**: Extract the correspondence from the zero set of the optimized $z$.

All plotting is routed through `plot_utils`, which has extra dependencies, this can be adapted to your likings.

## Setup

In [ ]:
import os
import sys

# ScalableDenseMaps ships with this repo and provides the precise-map data structure
# used to represent and apply correspondences.
sys.path.append(os.path.abspath("./ScalableDenseMaps/"))

from diffops import Pipeline

import plot_utils as plu

## Load the input meshes

We read the two surfaces $A$ and $B$, then convert them to `SurfaceMesh` objects and
compute an **intrinsic Delaunay** triangulation. 

In [ ]:
mesh_path_A = "./data/sitting_armadillo/A_1.obj"
mesh_path_B = "./data/sitting_armadillo/B_1.obj"

# The Pipeline holds the two surfaces and every intermediate quantity computed below.
pipe = Pipeline.from_files(mesh_path_A, mesh_path_B, verbose=True)

In [ ]:
pipe.mesh1.n_vertices, pipe.mesh2.n_vertices

In [ ]:
plu.plot(pipe.mesh1)

## Input correspondence

The method starts from an input map given as **vertex-to-face maps** in both directions.
Here we use a trivial nearest-neighbor (in ambient space) initialization on the original
triangulations.

In [ ]:
# Vertex-to-face precise maps on the ORIGINAL triangulations, converted internally into
# the per-face intrinsic vector fields used to build the initial section.
pipe.set_initial_map_nn()

In [ ]:
# Visualize the raw input map (mesh2 pushed onto mesh1's connectivity).
# plu.plot(plu.ToyMesh(pipe.P12_input @ pipe.mesh2.vertices, pipe.f1))
plu.plot_p2p(pipe.mesh1, pipe.mesh2, pipe.P21_input)

## Step (1): Construct surface connections

On each surface we build a discrete connection whose curvature is **half the Gaussian
curvature** (which integrates to $2\pi$ by Gauss-Bonnet). This forces exactly one zero per slice of $A \times B$.

In [ ]:
# The per-surface curvatures and connections are stored on the pipeline, and are
# available as `pipe.curvatures` and `pipe.connections`.
pipe.compute_connections()

## Step (2): Initialize the section

We assemble the product mesh $A \times B$ and compute an initial complex section whose
zero set approximates the graph of the input map. This reduces to an eigenvector problem, solved with lobpcg

In [ ]:
initial_section = pipe.initialize_section(use_preconditioner=True, verbose=True)

In [ ]:
# Extract and visualize the correspondence encoded by the initial section, to see the
# starting point of the optimization.
P21_init = pipe.extract_map(initial_section, direction="21", verbose=False)

# plu.plot(plu.ToyMesh(P21_init @ pipe.mesh1.vertices, pipe.f2))
plu.plot_p2p(pipe.mesh1, pipe.mesh2, P21_init)

## Step (3): Ginzburg-Landau minimization

We now minimize the discrete Ginzburg-Landau energy of the section.

The weight `lam` sets the strength of the circular-well potential (larger =
sharper zeros). Passing a *sequence* runs an annealing schedule where each stage
warm-starts from the previous one.

In [ ]:
# Assemble the product-space connection Laplacian and scalar mass operators.
pipe.build_operators()

In [ ]:
# Critical Ginzburg-Landau parameter from the smallest eigenvalues of L1, L2.
# It retuns epsilon but also caches it
pipe.compute_critical_epsilon()

In [ ]:
# `optimize` returns the optimized section; the per-stage energy histories are
# kept in `pipe._loggers` (and the last one in `pipe.logger`).
u_optimized = pipe.optimize(
    initial_section,
    lam=(1e2, 5e1),  # annealing schedule: align, then refine
    alpha=1.0,  # weight of the Dirichlet term
    n_iter=1000,
    log_interval=50,
    gtol=1e-8,
)

## Step (4): Evaluate the correspondence

Finally, we read the correspondence off the optimized section by locating, for each
vertex, the single zero of $z$ along its slice, and converting that intrinsic location
back to the original triangulation. We extract both directions and visualize the result.

In [ ]:
# Map from A to B and from B to A, wrapped as precise maps on the ORIGINAL triangulations
P21_final = pipe.extract_map(section=u_optimized, direction="21", verbose=False)
P12_final = pipe.extract_map(section=u_optimized, direction="12", verbose=False)


# Note that these maps are vertex to barycentric maps
# If you want to get the nearest neighbor map, you can call `get_nn()` on them:
# p2p_12 = P12_final.get_nn()
# If you want the exact face and barycentric, these are stored in `v2f_21` and `bary_coords` attributes.
P12_final.get_nn()

In [ ]:
# Visualize the final correspondence.
# plu.plot(plu.ToyMesh(P21_final @ pipe.mesh1.vertices, pipe.f2))
plu.plot_p2p(
    pipe.mesh1,
    pipe.mesh2,
    P21_final,
    uv1=pipe.mesh1.vertices[:, :2],
    texture="texture_1.png",
)

In [ ]:
# All four steps above are chained by `Pipeline.run`, which returns the final map.
# Every intermediate quantity stays available on the instance afterwards.

# P21_final = Pipeline.from_files(mesh_path_A, mesh_path_B).run()

## Landmarks

The circular-well potential can be adapted to landmarks. We there use the potential

$$V = \min_k \left( 1 - \exp\left( -\frac{d_A(\cdot, L^A_k)^2}{2\sigma^2}
                                      -\frac{d_B(\cdot, L^B_k)^2}{2\sigma^2} \right) \right),$$


Landmarks are read from a pair of `.pinned` files, in which line $k$ of each file
describes one landmark. A landmar is either a **single** vertex index is a point landmark, and
**several** indices form a curve landmark (points are then free to slide along the
curve). `Pipeline.from_files` picks these up automatically when they sit next to
the meshes.

In [ ]:
# hands_easy ships six point landmarks, loaded automatically from A/B.pinned.
pipe_lmk = Pipeline.from_files(
    "./data/hands_easy/A.obj",
    "./data/hands_easy/B.obj",
)
print(f"{len(pipe_lmk.landmarks)} landmark pairs")

# The pinning potential lives on the product space, flattened row-major.
V = pipe_lmk.compute_pin_matrix(pipe_lmk.landmarks)  # sigma defaults to pipe_lmk.sigma
V = V.reshape(pipe_lmk.mesh1.n_vertices, pipe_lmk.mesh2.n_vertices)

In [ ]:
# Landmarks are honoured by `run`/`optimize` as soon as they are set; `sigma`
# tunes how wide the wells are. Passing `landmarks=[]` disables them.
P21_lmk = pipe_lmk.run(direction="21", lam=(1e2, 5e1))

In [ ]:
plu.plot_p2p(pipe_lmk.mesh1, pipe_lmk.mesh2, P21_lmk)
# plu.plot(plu.ToyMesh(P21_lmk @ pipe_lmk.mesh1.vertices, pipe_lmk.f2))

In [ ]:
# Curve landmarks work the same way -- teddy mixes four point landmarks with two
# curves, and the curves need not have the same number of vertices on each side.
landmarks_teddy = Pipeline.read_pinned(
    "./data/teddy/A.pinned",
    "./data/teddy/B.pinned",
)
for lmk_A, lmk_B in landmarks_teddy:
    kind = "point" if len(lmk_A) == 1 else "curve"
    print(f"{kind:5s}  |A| = {len(lmk_A):2d}   |B| = {len(lmk_B):2d}")